# 19.1 Sources of complementarity

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Economic equilibria are one of the best-known applications of complementarity conditions.
We begin this section by showing how a previous linear programming example
in production economics has an equivalent form as a complementarity model, and how

```ampl
set PROD;                     # products
set ACT;                      # activities
param cost {ACT} > 0;     # cost per unit of each activity
param demand {PROD} >= 0; # units of demand for each product
param io {PROD,ACT} >= 0; # units of each product from
                            # 1 unit of each activity
var Level {j in ACT} >= 0;
minimize Total_Cost: sum {j in ACT} cost[j] * Level[j];
subject to Demand {i in PROD}:
       sum {j in ACT} io[i,j] * Level[j] >= demand[i];
```

**[Figure 19-1](./19_1_sources_of_complementarity.ipynb#fig-19-1):** Production cost minimization model (econmin.mod).

bounded variables are handled though an extension to the concept of complementarity.
We then describe a further extension to price-dependent demands that is not motivated by
optimization or equivalent to any linear program. We conclude by briefly describing
other complementarity models and applications.

**A complementarity model of production economics**

In [Section 2.4](../02/2_4_generalizations_to_blending_economics_and_scheduling.ipynb) we observed that the form of a diet model also applies to a model of
production economics. The decision variables may be taken as the levels of production
activities, so that the objective is the total production cost,

```ampl
minimize Total_Cost: sum {j in ACT} cost[j] * Level[j];
```

where cost[j] and Level[j] are the cost per unit and the level of activity j. The
constraints say that the totals of the product outputs must be at least the product demands:

```ampl
subject to Demand {i in PROD}:
       sum {j in ACT} io[i,j] * Level[j] >= demand[i];
```

with io[i,j] being the amount of product `i` produced per unit of activity j, and
demand[i] being the total quantity of product `i` demanded. Figures 19-1 and 19-2
show this "economic" model and some data for it.

Minimum-cost production levels are easily computed by a linear programming solver:

```ampl
ampl: model econmin.mod;
ampl: data econ.dat;
ampl: solve;
CPLEX 8.0.0: optimal solution; objective 6808640.553
3 dual simplex iterations (0 in phase I)
```

```ampl
param: ACT:         cost  :=
      P1          2450    P1a   1290
      P2          1850    P2a   3700    P2b   2150
      P3          2200    P3c   2370
      P4          2170   ;
param: PROD:        demand :=
     AA1           70000
     AC1           80000
     BC1           90000
     BC2           70000
     NA2          400000
     NA3          800000 ;
param io (tr):
       AA1 AC1    BC1   BC2     NA2    NA3 :=
 P1     60   20    10    15     938    295
 P1a     8    0    20    20    1180    770
 P2      8   10    15    10     945    440
 P2a    40   40    35    10     278    430
 P2b    15   35    15    15    1182    315
 P3     70   30    15    15     896    400
 P3c    25   40    30    30    1029    370
 P4     60   20    15    10    1397    450 ;
```

**[Figure 19-2](./19_1_sources_of_complementarity.ipynb#fig-19-2):** Data for production models (econ.dat).

```ampl
ampl: display Level;
Level [*] :=
       P1     0
P1a 1555.3
       P2     0
P2a     0
P2b     0
       P3   147.465
P3c 1889.4
       P4     0
```

Recall (from [Section 12.5](../12/12_5_displaying_solution_values.ipynb)) that there are also dual or marginal values — or "prices" —
associated with the constraints:

```ampl
ampl: display Demand.dual;
Demand.dual [*] :=
AA1 16.7051
AC1   5.44585
BC1 57.818
BC2   0
NA2   0
NA3   0
```

In the conventional linear programming interpretation, the price on constraint `i` gives,
within a sufficiently small range, the change in the total cost per unit change in the
demand for product i.

Consider now an alternative view of the production economics problem, in which we
define variables Price[i] as well as Level[j] and seek an equilibrium rather than
an optimum solution. There are two requirements that the equilibrium solution must satisfy.

First, for each product, total output must meet demand and the price must be nonnegative,
and in addition there must be a complementarity between these relationships: where
production exceeds demand the price must be zero, or equivalently, where the price is
positive the production must equal the demand. This relationship is expressed in AMPL
by means of the complements operator:

```ampl
subject to Pri_Compl {i in PROD}:
       Price[i] >= 0 complements
       sum {j in ACT} io[i,j] * Level[j] >= demand[i];
```

When two inequalities are joined by complements, they both must hold, and at least
one must hold with equality. Because our example is indexed over the set PROD, it sets
up a relationship of this kind for each product.

Second, for each activity, there is another relationship that may at first be less obvious.
Consider that, for each unit of activity j, the value of the resulting product `i` output
in terms of the model's prices is Price[i] * io[i,j]. The total value of all outputs
from one unit of activity `j` is thus

```ampl
sum {i in ACT} Price[i] * io[i,j]
```

At equilibrium prices, this total value cannot exceed the activity's cost per unit,
cost[j]. Moreover, there is a complementarity between this relationship and the level
of activity j: where cost exceeds total value the activity must be zero, or equivalently,
where the activity is positive the total value must equal the cost. Again this relationship
can be expressed in AMPL with the complements operator:

```ampl
subject to Lev_Compl {j in ACT}:
       Level[j] >= 0 complements
       sum {i in PROD} Price[i] * io[i,j] <= cost[j];
```

Here the constraint is indexed over ACT, so that we have a complementarity relationship
for each activity.

Putting together the two collections of complementarity constraints, we have the linear
complementarity problem shown in [Figure 19-3](./19_1_sources_of_complementarity.ipynb#fig-19-3). The number of variables and the
number of complementarity relationships are equal (to activities plus products), making
this a "square" complementarity problem that is amenable to certain solution techniques,
though not the same techniques as those for linear programs.

Applying the PATH solver, for example, the complementarity problem can be seen to
have the same solution as the related minimum-cost production problem:

```ampl
set PROD;                       # products
set ACT;                        # activities
param cost {ACT} > 0;           #    cost per unit of each activity
param demand {PROD} >= 0;       #    units of demand for each product
param io {PROD,ACT} >= 0;       #    units of each product from
                                   #    1 unit of each activity
var Price {i in PROD};
var Level {j in ACT};
subject to Pri_Compl {i in PROD}:
       Price[i] >= 0 complements
       sum {j in ACT} io[i,j] * Level[j] >= demand[i];
subject to Lev_Compl {j in ACT}:
       Level[j] >= 0 complements
       sum {i in PROD} Price[i] * io[i,j] <= cost[j];
```

**[Figure 19-3](./19_1_sources_of_complementarity.ipynb#fig-19-3):** Production equilibrium model (econ.mod).

```ampl
ampl: model econ.mod;
ampl: data econ.dat;
ampl: option solver path;
ampl: solve;
Path v4.5: Solution found.

7 iterations (0 for crash); 33 pivots.

20 function, 8 gradient evaluations.
ampl: display sum {j in ACT} cost[j] * Level[j];
sum{j in ACT} cost[j]*Level[j] = 6808640
```

Further application of display shows that Level is the same as in the production economics
LP and that Price takes the same values that Demand.dual has in the LP.

**Complementarity for bounded variables**

Suppose now that we extend our models by placing bounds on the activity levels:
level_min[j] <= Level[j] <= level_max[j]. The equivalence between the
optimization problem and a square complementarity problem can be maintained, provided
that the complementarity relationship for the activities is generalized to a "mixed"
form. Where an activity's cost is greater than its total value (per unit), the activity's level
must be at its lower bound (much as before). Where an activity's level is between its
bounds, its cost must equal its total value. And an activity's cost may also be less than its
total value, provided that its level is at its upper bound. These three relationships are
summarized by another form of the complements operator:

```ampl
subject to Lev_Compl {j in ACT}:
       level_min[j] <= Level[j] <= level_max[j] complements
       cost[j] - sum {i in PROD} Price[i] * io[i,j];
```

```ampl
set PROD;                       # products
set ACT;                        # activities
param cost {ACT} > 0;           #    cost per unit of each activity
param demand {PROD} >= 0;       #    units of demand for each product
param io {PROD,ACT} >= 0;       #    units of each product from
                                   #    1 unit of each activity
param level_min {ACT} > 0; # min allowed level for each activity
param level_max {ACT} > 0; # max allowed level for each activity
var Price {i in PROD};
var Level {j in ACT};
subject to Pri_Compl {i in PROD}:
       Price[i] >= 0 complements
       sum {j in ACT} io[i,j] * Level[j] >= demand[i];
subject to Lev_Compl {j in ACT}:
       level_min[j] <= Level[j] <= level_max[j] complements
       cost[j] - sum {i in PROD} Price[i] * io[i,j];
```

**[Figure 19-4](./19_1_sources_of_complementarity.ipynb#fig-19-4):** Bounded version of production equilibrium model (econ2.mod).

When a double inequality is joined to an expression by complements, the inequalities
must hold, and either the expression must be zero, or the lower inequality must hold with
equality and the expression must be nonnegative, or the upper inequality must hold with
equality and the expression must be nonpositive.

A bounded version of our complementarity examples is shown in [Figure 19-4](./19_1_sources_of_complementarity.ipynb#fig-19-4). The
PATH solver can be applied to this model as well:

```ampl
ampl: model econ2.mod;
ampl: data econ2.dat;
ampl: option solver path;
ampl: solve;
Path v4.5: Solution found.

9 iterations (4 for crash); 8 pivots.

22 function, 10 gradient evaluations.

ampl: display level_min, Level, level_max;
:   level_min Level level_max :=
P1      240      240    1000
P1a     270     1000    1000
P2      220      220    1000
P2a     260      680    1000
P2b     200      200    1000
P3      260      260    1000
P3c     220     1000    1000
P4      240      240    1000
;
```

The results are the same as for the LP that is derived from our previous example ([Figure 19-1](./19_1_sources_of_complementarity.ipynb#fig-19-1))
by adding the bounds above to the variables.

```ampl
set PROD;                       # products
set ACT;                        # activities
param cost {ACT} > 0;           # cost per unit of each activity
param io {PROD,ACT} >= 0;       # units of each product from
                                   # 1 unit of each activity
param demzero {PROD} > 0; # intercept and slope of the demand
param demrate {PROD} >= 0; # as a function of price
var Price {i in PROD};
var Level {j in ACT};
subject to Pri_Compl {i in PROD}:
       Price[i] >= 0 complements
       sum {j in ACT} io[i,j] * Level[j]
              >= demzero[i] - demrate[i] * Price[i];
subject to Lev_Compl {j in ACT}:
       Level[j] >= 0 complements
       sum {i in PROD} Price[i] * io[i,j] <= cost[j];
```

**[Figure 19-5](./19_1_sources_of_complementarity.ipynb#fig-19-5):** Price-dependent demands (econnl.mod).

**Complementarity for price-dependent demands**

If complementarity problems only arose from linear programs, they would be of very
limited interest. The idea of an economic equilibrium can be generalized, however, to
problems that have no LP equivalents. Rather than taking the demands to be fixed, for
example, it makes sense to view the demand for each product as a decreasing function of
its price.

The simplest case is a decreasing linear demand, which could be expressed in AMPL
as

```ampl
demzero[i] - demrate[i] * Price[i]
```

where demzero[i] and demrate[i] are nonnegative parameters. The resulting
complementarity problem simply substitutes this expression for demand[i], as seen in
[Figure 19-5](./19_1_sources_of_complementarity.ipynb#fig-19-5). The complementarity problem remains square, and can still be solved by
PATH, but with clearly different results:

```ampl
ampl: model econnl.mod;
ampl: data econnl.dat;
ampl: option solver path;
ampl: solve;
Path v4.5: Solution found.

11 iterations (3 for crash); 11 pivots.
12 function, 12 gradient evaluations.
ampl: display Level;
Level [*] :=
P1 240
P1a 710.156
P2 220
P2a 260
P2b 200
P3 260
P3c 939.063
P4 240
;
```

The balance between demands and prices now tends to push down the equilibrium production
levels.

Because the Price[i] variables appear on both sides of the complements operator
in this model, there is no equivalent linear program. There does exist an equivalent
nonlinear optimization model, but it is not as easy to derive and may be harder to solve as
well.

**Other complementarity models and applications**

This basic example can be extended to considerably more complex models of economic
equilibrium. The activity and price variables and their corresponding complementarity
constraints can be comprised of several indexed collections each, and both the cost
and price functions can be nonlinear. A solver such as PATH handles all of these extensions,
so long as the problem remains square in the sense of having equal numbers of
variables and complementarity constraints (or being easily converted to such a form as
explained in the next section).

More ambitious models may add an objective function and may mix equality, inequality
and complementarity constraints in arbitrary numbers. Solution techniques for these
so-called MPECs — mathematical programs with equilibrium constraints — are at a relatively
experimental stage, however.

Complementarity problems also arise in physical systems, where they can serve as
models of equilibrium conditions between forces. A complementarity constraint may
represent a discretization of the relationship between two objects, for example. The relationship
on one side of the complements operator may hold with equality at points
where the objects are in contact, while the relationship on the other side holds with equality
where they do not touch.

Game theory provides another class of examples. The Nash equilibrium for a bimatrix
game is characterized by complementarity conditions, for example, in which the
variables are the probabilities with which the two players make their available moves.
For each move, either its probability is zero, or a related equality holds to insure there is
nothing to be gained by increasing or decreasing its probability.

Surveys that describe a variety of complementarity problems in detail are cited in the
references at the end of this chapter.